In [134]:
from tensorflow.keras.datasets import imdb

In [135]:
number_of_words = 10000

In [136]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=number_of_words)

In [137]:
#Data Exploration

In [138]:
X_train.shape           #Training set samples

(25000,)

In [139]:
y_train.shape       #Training set labels

(25000,)

In [140]:
X_test.shape        #Testint set samples

(25000,)

In [141]:
y_test.shape        #Testing set labels

(25000,)

In [142]:
%pprint

Pretty printing has been turned OFF


In [143]:
X_train[123]        #Samples as numeric data

[1, 307, 5, 1301, 20, 1026, 2511, 87, 2775, 52, 116, 5, 31, 7, 4, 91, 1220, 102, 13, 28, 110, 11, 6, 137, 13, 115, 219, 141, 35, 221, 956, 54, 13, 16, 11, 2714, 61, 322, 423, 12, 38, 76, 59, 1803, 72, 8, 2, 23, 5, 967, 12, 38, 85, 62, 358, 99]

In [144]:
#Decoding a Movie reviewed

In [145]:
word_to_index = imdb.get_word_index()

In [146]:
word_to_index['great']      #Show the index (84th) most frequent word

84

In [147]:
index_to_word = {           #Reverse the word_to_index dictionary's mapping
    index:word for (word, index) in word_to_index.items()
}

In [148]:
[index_to_word[i] for i in range(1,51)]

['the', 'and', 'a', 'of', 'to', 'is', 'br', 'in', 'it', 'i', 'this', 'that', 'was', 'as', 'for', 'with', 'movie', 'but', 'film', 'on', 'not', 'you', 'are', 'his', 'have', 'he', 'be', 'one', 'all', 'at', 'by', 'an', 'they', 'who', 'so', 'from', 'like', 'her', 'or', 'just', 'about', "it's", 'out', 'has', 'if', 'some', 'there', 'what', 'good', 'more']

In [149]:
' '.join([index_to_word.get(i-3,'?') for i in X_train[123]])

'? beautiful and touching movie rich colors great settings good acting and one of the most charming movies i have seen in a while i never saw such an interesting setting when i was in china my wife liked it so much she asked me to ? on and rate it so other would enjoy too'

In [150]:
y_train[123]            #The revbies is classified as positive

np.int64(1)

In [151]:
#Data Preparation

In [152]:
words_per_review = 200

In [153]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [154]:
X_train = pad_sequences(X_train, maxlen=words_per_review)       #Return a 2D array with the number of features specified by the maxlen argument

In [155]:
X_train.shape

(25000, 200)

In [156]:
X_test = pad_sequences(X_test,maxlen=words_per_review)  #Return a 2D array with the number of features specified by the maxlen argument

In [157]:
X_test.shape

(25000, 200)

In [158]:
#Splitting the Test Data into validation and Test Data

In [159]:
from sklearn.model_selection import train_test_split

In [160]:
X_test, X_val, y_test, y_val = train_test_split(
    X_test,
    y_test,
    random_state=11,
    test_size=0.2,
)

In [161]:
X_test.shape

(20000, 200)

In [162]:
X_val.shape

(5000, 200)

In [163]:
#Creating the Neural Network

In [164]:
from tensorflow.keras.models import Sequential

In [165]:
rnn = Sequential()

In [166]:
#Import the layers

In [167]:
from tensorflow.keras.layers import Dense, LSTM

In [168]:
from tensorflow.keras.layers import Embedding

In [169]:
#Embedding layer to reduce data

In [170]:
rnn.add(
    Embedding(
    input_dim=number_of_words,              #Number of unique words
    output_dim=128,                         #Size of each word embedding
   input_length=words_per_review)          #Number of words in each input sample
)

In [171]:
#Adding an LTSM Layer

In [172]:
rnn.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [173]:
rnn.add(
    LSTM(
        units=128,                  #Number of neurons in the layer
        dropout=0.2,
        recurrent_dropout=0.2
    )
)

In [174]:
#Adding a Dense Output Layer

In [175]:
rnn.add(
    Dense(
        units=1,                    #Reduce to one result indicating whether a review is positive or negative
        activation='sigmoid'        #Reduces arbitrary values into the range 0.0-1.0
    )
)

In [176]:
#Compile the model and displaying the summary

In [177]:
rnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',         #There're only two possible outputs
    metrics=['accuracy']
)

In [178]:
rnn.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [179]:
#Training and Evaluating the model

In [180]:
rnn.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 99s 124ms/step - accuracy: 0.7648 - loss: 0.4883 - val_accuracy: 0.8166 - val_loss: 0.4270
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 103s 132ms/step - accuracy: 0.8464 - loss: 0.3637 - val_accuracy: 0.8259 - val_loss: 0.4015
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 108s 139ms/step - accuracy: 0.8848 - loss: 0.2917 - val_accuracy: 0.8008 - val_loss: 0.4383
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 114s 146ms/step - accuracy: 0.9059 - loss: 0.2425 - val_accuracy: 0.8574 - val_loss: 0.3802
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 116s 148ms/step - accuracy: 0.9185 - loss: 0.2083 - val_accuracy: 0.8657 - val_loss: 0.3569
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 118s 150ms/step - accuracy: 0.9480 - loss: 0.1409 - val_accuracy: 0.8535 - val_loss: 0.4153
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 117s 150ms/step - accuracy: 0.9637 - loss: 0.1024 - val_accuracy: 0.8574 - val_loss: 0.4236
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 120s 153ms/step - accuracy: 0.9698 - 

In [181]:
results = rnn.evaluate(X_test, y_test)      #Evaluate the results using the test data. Returns the loss and accuracy values

625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 24ms/step - accuracy: 0.8541 - loss: 0.6372
